# 5회차 실습 — AI Agent Web Service MVP 완성 및 평가

## 학습목표
- Web UI–FastAPI–Agent–LLM–Tool 전체 구조를 연결한다.
- 브라우저에서 Agent를 실행하는 최종 MVP를 완성한다.
- 계층별로 오류를 추적하고 수정한다.
- 응답시간과 기능 테스트 결과를 기록한다.
- GitHub에 최종 프로젝트와 개인 역할을 정리한다.

### 실행 구분
- 🟢 LOCAL: HTML/CSS/JS, 코드 검토, Codex 수정
- 🔴 GPU: Qwen3-8B 실제 Agent 실행
- 📸 CAPTURE: 최종 포트폴리오 캡처 권장

> 최종 구조: **Web UI → FastAPI → Agent → Qwen3-8B → Custom Tool → Web UI**


## Step 1. 4회차 프로젝트를 5회차 MVP로 확장 — 🟢 LOCAL

```text
07_agent_web_mvp/
├─ app/
│  ├─ tools.py
│  ├─ llm.py
│  ├─ agent.py
│  ├─ main.py
│  └─ static/
│     ├─ index.html
│     ├─ style.css
│     └─ app.js
├─ requirements.txt
└─ README.md
```

4회차의 FastAPI Agent Service에 **브라우저 UI 계층**을 추가합니다.


## Step 2. Web UI 역할 이해 — 🟢 LOCAL

- `index.html`: 화면 구조
- `style.css`: 화면 디자인
- `app.js`: `/agent`, `/health` API 호출

Web UI에는 LLM이나 Tool 로직을 넣지 않습니다.  
**UI는 입력과 출력**, 실제 AI 처리는 서버가 담당합니다.


## Step 3. JavaScript의 Agent API 호출 — 🟢 LOCAL / 📸

```javascript
const res = await fetch("/agent", {
  method: "POST",
  headers: {"Content-Type":"application/json"},
  body: JSON.stringify({query})
});

const data = await res.json();
answerEl.textContent = data.final_answer;
```

브라우저는 HTTP를 통해 FastAPI와 통신합니다.


## Step 4. FastAPI에서 Web UI 제공 — 🟢 LOCAL

`main.py`에서 정적 파일과 `index.html`을 제공합니다.

```python
app.mount("/static", StaticFiles(directory=STATIC_DIR), name="static")

@app.get("/")
def web_ui():
    return FileResponse(STATIC_DIR / "index.html")
```


## Step 5. 전체 계층 연결 확인 — 📸

```text
Browser
  ↓ fetch("/agent")
FastAPI main.py
  ↓
run_agent()
  ↓
Qwen3-8B 판단
  ↓
Custom Tool 실행
  ↓
Qwen3-8B 최종 응답
  ↓
JSON Response
  ↓
Browser 화면 표시
```


## Step 6. Codex로 UI 코드 검토 — 🟢 LOCAL

### 프롬프트
```text
이 프로젝트는 교육용 AI Agent Web MVP다.
app/static/index.html, style.css, app.js를 검토해줘.

1. /agent 호출 흐름
2. 사용자 입력 검증
3. 오류 처리
4. 학생이 이해하기 어려운 부분

을 설명해줘. 아직 코드는 수정하지 마.
```


## Step 7. Codex 미션 — UI 기능 개선 — 🟢 LOCAL

팀별로 하나를 선택합니다.

- Enter/Ctrl+Enter 실행
- 예시 질문 버튼
- 실행 이력 표시
- Tool 사용 여부 Badge
- elapsed_ms 표시 개선
- 오류 메시지 개선

### 지시 원칙
**목표 / 수정 허용 파일 / 유지 조건 / 완료 조건**을 명시합니다.


## Step 8. 로컬 사전 검증 — 🟢 LOCAL

GPU를 켜기 전에:

```bash
python -m compileall app
git status
git diff
```

HTML/CSS/JS는 코드 검토와 브라우저 개발자 도구를 활용합니다.


## Step 9. RunPod 시작 및 GPU 확인 — 🔴 GPU

```bash
nvidia-smi
```

확인:
- A100 할당 여부
- CUDA 사용 가능 여부
- 불필요한 CPU fallback 사용 여부

GPU가 정상일 때만 모델 추론 단계로 진행합니다.


## Step 10. 프로젝트 동기화 — 🔴 GPU

```bash
cd /root/yu-gpu-ai
git pull

cd 07_agent_web_mvp
pip install -r requirements.txt
```


## Step 11. 최종 MVP 실행 — 🔴 GPU

```bash
uvicorn app.main:app --host 0.0.0.0 --port 8000
```

RunPod의 HTTP Service/프록시를 통해 브라우저로 접속합니다.

> URL 형식은 실제 Pod 설정에 따라 달라질 수 있으므로 현재 화면에서 확인합니다.


## Step 12. 기능 테스트 ① GPU 상태 — 🔴 GPU / 📸

Web UI의 **GPU 상태** 버튼을 실행합니다.

확인:
- `cuda_available = true`
- GPU 이름
- 모델 ID

Agent 질문을 보내기 전에 인프라 상태부터 확인합니다.


## Step 13. 기능 테스트 ② 계산 Tool — 🔴 GPU / 📸

질문:
```text
33명이 30시간씩 GPU를 사용하면 총 몇 GPU-hour야?
```

확인:
- `used_tool = true`
- `tool_name = calculator`
- 계산 결과
- 최종 자연어 응답
- `elapsed_ms`


## Step 14. 기능 테스트 ③ text_length — 🔴 GPU

질문:
```text
'AI Agent Web Service MVP'의 글자 수를 알려줘.
```

Agent가 `text_length`를 선택하는지 확인합니다.


## Step 15. 기능 테스트 ④ 일반 설명 — 🔴 GPU

질문:
```text
AI Agent와 일반 챗봇의 차이를 한 문장으로 설명해줘.
```

확인:
- Tool을 불필요하게 호출하지 않는가?
- `action = answer`로 직접 응답하는가?


## Step 16. 계층별 디버깅 — 📸

| 증상 | 먼저 확인할 계층 |
|---|---|
| 화면이 안 뜸 | HTML / StaticFiles / `/` |
| 버튼이 반응 없음 | JavaScript |
| 404/422 | FastAPI route / Request Body |
| CUDA False | RunPod / llm.py |
| Tool 선택 오류 | agent.py / Prompt |
| Tool 실행 오류 | tools.py |
| JSON 파싱 오류 | parse_agent_json |
| 답변 표시 오류 | app.js |


## Step 17. 성능 측정 — 🔴 GPU

5회차에서는 서비스 관점에서 **End-to-End 응답시간**을 확인합니다.

`run_agent()` 결과의:
```json
{
  "elapsed_ms": 5230.4
}
```

이 값은 LLM 판단, Tool 실행, 최종 응답 생성 등을 포함한 Agent 전체 실행시간입니다.

> 2회차의 Tokens/sec와 목적이 다릅니다.


## Step 18. MVP 평가 데이터 작성 — 🟢 LOCAL

팀별로 최소 5개 테스트를 기록합니다.

```python
test_cases = [
    {"type":"계산", "query":"..."},
    {"type":"글자수", "query":"..."},
    {"type":"시간", "query":"..."},
    {"type":"일반설명", "query":"..."},
    {"type":"오류/경계", "query":"..."},
]
```

기록 항목:
- 기능 성공 여부
- 선택 Tool
- 응답시간
- 오류 내용
- 개선사항


In [ ]:
import pandas as pd

evaluation_columns = [
    "test_type", "query", "success",
    "used_tool", "tool_name",
    "elapsed_ms", "issue", "improvement"
]

mvp_evaluation_df = pd.DataFrame(columns=evaluation_columns)
mvp_evaluation_df


## Step 19. MVP 평가표 저장 — 🟢 LOCAL

실제 테스트 결과를 입력한 뒤 저장합니다.


In [ ]:
from pathlib import Path

results_dir = Path.cwd() / "results"
results_dir.mkdir(parents=True, exist_ok=True)

mvp_path = results_dir / "mvp_evaluation.csv"
mvp_evaluation_df.to_csv(mvp_path, index=False, encoding="utf-8-sig")

print("저장 위치 :", mvp_path)


## Step 20. 팀 최종 개선 — 🟢 LOCAL + 🔴 GPU

평가 결과에서 가장 중요한 문제 **1개만 선택**하여 수정합니다.

예:
- Tool Description 개선
- JSON 파싱 안정성 개선
- 오류 메시지 개선
- UI 사용성 개선
- 응답시간 표시 개선

수정 후 동일 테스트를 다시 실행하여 **Before → After**를 기록합니다.


## Step 21. GitHub 최종 정리 — 🟢 LOCAL

```bash
git status
git diff

git add .
git commit -m "Complete AI agent web service MVP"
git push
```

README에는 다음을 포함합니다.
1. 프로젝트 목적
2. 아키텍처
3. 실행 방법
4. 주요 기능
5. 테스트 결과
6. 팀원 역할


## Step 22. 개인 포트폴리오 기록

각 학생은 팀 결과와 별도로 자신의 기여를 기록합니다.

- 담당 기능
- 수정한 파일
- 해결한 오류
- 사용한 평가 지표
- Before / After
- Git commit 또는 코드 근거
- 배운 점

**“팀 프로젝트에 참여했다”보다 “내가 무엇을 구현·검증했는가”가 중요합니다.**


## Step 23. 최종 발표 체크리스트

### 5분 발표 예시
1. 문제 정의 — 30초
2. 서비스 구조 — 60초
3. 핵심 기능 시연 — 90초
4. 평가 결과 — 60초
5. 개선 내용 — 30초
6. 팀원 역할 — 30초

시연이 실패할 경우를 대비해 정상 실행 화면 캡처도 준비합니다.


## 5회차 실습 완료 — 최종 산출물

- `07_AI_Agent_Web_MVP.ipynb`
- `07_agent_web_mvp/` 전체 소스
- Web UI + FastAPI + Agent + Qwen + Custom Tool
- `results/mvp_evaluation.csv`
- README
- GitHub 최종 commit
- 개인 역할·기여 기록

### 5일 프로젝트 흐름
**GPU LLM 실행 → 모델 평가 → Custom Tool Agent → FastAPI Service → Web MVP + 평가**
